In [ ]:
import pyzed.sl as sl
import cv2
import numpy as np
import threading
import ipywidgets as widgets
from IPython.display import display
import time

# Video Display Widget Setup
display_color = widgets.Image(format='jpeg', width='45%')
display_depth = widgets.Image(format='jpeg', width='45%')
layout = widgets.Layout(width='100%')
sidebyside = widgets.HBox([display_color, display_depth], layout=layout)

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

# Camera Setup
class Camera():
    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA 
        init_params.depth_mode = sl.DEPTH_MODE.ULTRA
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open Error: " + repr(status))
            exit(1)
            
        self.runtime = sl.RuntimeParameters()
        
        # Enable SVO2 Recording
        recording_param = sl.RecordingParameters('driving_data_3.svo2', sl.SVO_COMPRESSION_MODE.LOSSLESS)
        err = self.zed.enable_recording(recording_param)
        if err != sl.ERROR_CODE.SUCCESS:
            print("Recording Error: ", err)
            exit(1)
            
        self.thread_runnning_flag = False
        
        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width 
        self.height = camera_info.camera_configuration.resolution.height 
        
        self.image = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)
        self.depth = sl.Mat(self.width, self.height, sl.MAT_TYPE.F32_C1, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                self.zed.retrieve_measure(self.depth, sl.MEASURE.DEPTH)
                
                scale = 0.5
                color_value = self.image.get_data()
                resized_color = cv2.resize(color_value, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
                display_color.value = bgr8_to_jpeg(resized_color)
                
                depth_image = np.asanyarray(self.depth.get_data())
                depth_colormap = cv2.applyColorMap(cv2.convertScaleAbs(depth_image, alpha=0.03), cv2.COLORMAP_JET)
                resized_depth = cv2.resize(depth_colormap, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
                display_depth.value = bgr8_to_jpeg(resized_depth)
                        
    def start(self): 
        if self.thread_runnning_flag == False: 
            self.thread_runnning_flag = True 
            self.thread = threading.Thread(target=self._capture_frames) 
            self.thread.start() 

    def stop(self): 
        if self.thread_runnning_flag == True:
            self.thread_runnning_flag = False 
            self.thread.join()
            self.zed.disable_recording()
            self.zed.close()
            print("Recording saved successfully.")

display(sidebyside)
camera = Camera()


[2026-03-12 13:04:03 UTC][ZED][INFO] Logging level INFO
[2026-03-12 13:04:03 UTC][ZED][INFO] Logging level INFO
[2026-03-12 13:04:03 UTC][ZED][INFO] Logging level INFO
[2026-03-12 13:04:04 UTC][ZED][INFO] [Init]  Depth mode: ULTRA
[2026-03-12 13:04:05 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-03-12 13:04:05 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-03-12 13:04:05 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-03-12 13:04:05 UTC][ZED][INFO] [Init]  Serial Number: S/N 39784002
[2026-03-12 13:04:05 UTC][ZED][INFO] [Init]  Notice: The recording is using SVO version 2, enabled by default starting from SDK version 4.1. To revert to the original SVO version, set the environment variable "ZED_SDK_SVO_VERSION" to 1
Recording started


In [ ]:
time.sleep(10)
camera.start()
print("Recording started")

time.sleep(30)
camera.stop()